# T-Lab RecSys 2026: Causal GPT-2 Sequential Recommender System
### End-to-End Training on Kaggle with NVIDIA GPU (T4 / P100) & Mixed Precision (FP16)

This notebook trains a **Causal GPT-2 Transformer from scratch** for Next-Item Prediction on the **Yambda** dataset (Yandex Music).

**Pre-flight Checklist on Kaggle:**
1. In the right sidebar under **Settings -> Accelerator**, select **GPU T4 x 2** or **GPU P100**.
2. In **Settings -> Internet**, toggle **Internet ON**.
3. Attach your Yambda dataset under `/kaggle/input/` (via "+ Add Input" in the top right), or run as-is for synthetic demo mode.

## 1. Verify GPU Accelerator

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Workspace & Codebase Setup (Self-Contained)
This cell ensures that `train_base.py`, `data_utils.py`, `dataset.py`, and `model.py` are present in the active directory and registered in Python's `sys.path`. It works out of the box whether you cloned via Git or uploaded this notebook directly!

In [ ]:
import os
import sys
import base64

# 1. Check if user cloned the repo into a subfolder (using %cd magic)
subdirs = [
    "/kaggle/working/music-recommender-system",
    "/kaggle/working/Music-Recommender-System",
]
for sd in subdirs:
    if os.path.exists(os.path.join(sd, "train_base.py")):
        print(f"Found cloned repo at {sd}, switching directory...")
        %cd {sd}
        break

# 2. Self-Contained Deployment: If Python modules are not present, unpack them automatically
EMBEDDED_MODULES = {
    "data_utils.py": "...",
    "dataset.py": "IiIiCmRhdGFzZXQucHkKLS0tLS0tLS0tLQpQeVRvcmNoIERhdGFzZXQgYW5kIERhdGFMb2FkZXIgZm9yIEF1dG9yZWdyZXNzaXZlIFNlcXVlbnRpYWwgUmVjb21tZW5kZXIgU3lzdGVtcyAoTmV4dCBJdGVtIFByZWRpY3Rpb24pLgoKSW1wbGVtZW50czoKMS4gQ2hyb25vbG9naWNhbCBhZ2dyZWdhdGlvbiBwZXIgdXNlciAoc29ydGVkIGJ5IHRpbWVzdGFtcCkuCjIuIEZpeGVkLWxlbmd0aCBzZXF1ZW5jZSB3aW5kb3dpbmcgKG1heF9sZW5ndGgpLgozLiBDYXVzYWwgbGFuZ3VhZ2UgbW9kZWxpbmcgdGFyZ2V0IGFsaWdubWVudDoKICAgLSBJbnB1dCBTZXF1ZW5jZTogIFtzXzEsIHNfMiwgLi4uLCBzX2tdCiAgIC0gU2hpZnRlZCBMYWJlbDogICBbc18yLCBzXzMsIC4uLiwgc197aysxfV0KNC4gQXR0ZW50aW9uIG1hc2tpbmcgYW5kIHByb3BlciB6ZXJvLXBhZGRpbmcgKDxQQUQ+ID0gMCkuCjUuIExpc3RlbiBkdXJhdGlvbiBhbGlnbm1lbnQgW3JfMiwgcl8zLCAuLi4sIHJfe2srMX1dIHRvIHN1cHBvcnQgUGhhc2UgMiBSTCByZXdhcmQgZmluZS10dW5pbmcuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUsIFVuaW9uCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBEYXRhc2V0Cgpsb2dnaW5nLmJhc2ljQ29uZmlnKAogICAgbGV2ZWw9bG9nZ2luZy5JTkZPLAogICAgZm9ybWF0PSJbJShhc2N0aW1lKXNdIFslKGxldmVsbmFtZSlzXSAoJShuYW1lKXMpICUobWVzc2FnZSlzIiwKICAgIGRhdGVmbXQ9IiVZLSVtLSVkICVIOiVNOiVTIiwKKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiRGF0YXNldCIpCgoKQGRhdGFjbGFzcwpjbGFzcyBSZWNCYXRjaDoKICAgICIiIgogICAgU3RydWN0dXJlZCBQeVRvcmNoIEJhdGNoIGZvciBDYXVzYWwgU2VxdWVudGlhbCBSZWNvbW1lbmRhdGlvbi4KICAgIAogICAgQXR0cmlidXRlczoKICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgaXRlbSB0b2tlbiBpbmRpY2VzLgogICAgICAgIGxhYmVsczogTG9uZ1RlbnNvciBbQiwgTF0gY29udGFpbmluZyBuZXh0LWl0ZW0gdGFyZ2V0IHRva2VuIGluZGljZXMuCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgMSBmb3IgdmFsaWQgaXRlbXMsIDAgZm9yIDxQQUQ+LgogICAgICAgIGR1cmF0aW9uczogRmxvYXRUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgbGlzdGVuaW5nIGR1cmF0aW9uIGluIHNlY29uZHMgKFJMIHJld2FyZHMpLgogICAgICAgIHVzZXJfaWRzOiBMaXN0W1VuaW9uW3N0ciwgaW50XV0gY29udGFpbmluZyByYXcgdXNlciBpZGVudGlmaWVycyBmb3IgZXZhbHVhdGlvbiBncm91cGluZy4KICAgICIiIgogICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IKICAgIGxhYmVsczogdG9yY2guVGVuc29yCiAgICBhdHRlbnRpb25fbWFzazogdG9yY2guVGVuc29yCiAgICBkdXJhdGlvbnM6IHRvcmNoLlRlbnNvcgogICAgdXNlcl9pZHM6IExpc3RbVW5pb25bc3RyLCBpbnRdXQoKICAgIGRlZiB0byhzZWxmLCBkZXZpY2U6IFVuaW9uW3RvcmNoLmRldmljZSwgc3RyXSkgLT4gUmVjQmF0Y2g6CiAgICAgICAgIiIiVHJhbnNmZXJzIGJhdGNoIHRlbnNvcnMgdG8gdGFyZ2V0IGNvbXB1dGUgZGV2aWNlIChDUFUvR1BVL01QUykuIiIiCiAgICAgICAgcmV0dXJuIFJlY0JhdGNoKAogICAgICAgICAgICBpbnB1dF9pZHM9c2VsZi5pbnB1dF9pZHMudG8oZGV2aWNlKSwKICAgICAgICAgICAgbGFiZWxzPXNlbGYubGFiZWxzLnRvKGRldmljZSksCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPXNlbGYuYXR0ZW50aW9uX21hc2sudG8oZGV2aWNlKSwKICAgICAgICAgICAgZHVyYXRpb25zPXNlbGYuZHVyYXRpb25zLnRvKGRldmljZSksCiAgICAgICAgICAgIHVzZXJfaWRzPXNlbGYudXNlcl9pZHMsCiAgICAgICAgKQoKCmNsYXNzIFNlcXVlbnRpYWxSZWNEYXRhc2V0KERhdGFzZXQpOgogICAgIiIiCiAgICBQeVRvcmNoIERhdGFzZXQgZ2VuZXJhdGluZyBjYXVzYWwgKGlucHV0LCBsYWJlbCkgc2VxdWVuY2UgcGFpcnMgZm9yIE5leHQgSXRlbSBQcmVkaWN0aW9uLgoKICAgIEFyZ3M6CiAgICAgICAgZGY6IERhdGFGcmFtZSBjb250YWluaW5nIGludGVyYWN0aW9uIGxvZ3MuIE11c3QgY29udGFpbiB1c2VyX2NvbCwgaXRlbV90b2tlbl9jb2wsCiAgICAgICAgICAgIHRpbWVzdGFtcF9jb2wsIGFuZCBvcHRpb25hbGx5IGR1cmF0aW9uX2NvbC4KICAgICAgICBtYXhfbGVuZ3RoOiBNYXhpbXVtIHNlcXVlbmNlIGxlbmd0aCAoY29udGV4dCB3aW5kb3cpIGZvciB0aGUgR1BULTIgbW9kZWwuCiAgICAgICAgdXNlcl9jb2w6IENvbHVtbiBuYW1lIGZvciB1c2VyIGlkZW50aWZpZXJzLgogICAgICAgIGl0ZW1fdG9rZW5fY29sOiBDb2x1bW4gbmFtZSBmb3IgbWFwcGVkIGNvbnRpbnVvdXMgaXRlbSB0b2tlbnMgKDEgLi4uIE4pLgogICAgICAgIHRpbWVzdGFtcF9jb2w6IENvbHVtbiBuYW1lIGZvciBVTklYIHRpbWVzdGFtcHMuCiAgICAgICAgZHVyYXRpb25fY29sOiBDb2x1bW4gbmFtZSBmb3IgbGlzdGVuIGR1cmF0aW9uIGluIHNlY29uZHMuCiAgICAgICAgc2xpZGluZ193aW5kb3c6IElmIFRydWUsIHVzZXMgc2xpZGluZyB3aW5kb3cgY2h1bmtpbmcgdG8gZXh0cmFjdCBtdWx0aXBsZSBzdWJzZXF1ZW5jZXMKICAgICAgICAgICAgZnJvbSBsb25nIHVzZXIgaGlzdG9yaWVzIChyZWNvbW1lbmRlZCBmb3IgdHJhaW5pbmcgZGF0YSBhdWdtZW50YXRpb24pLgogICAgICAgICAgICBJZiBGYWxzZSwgcmV0YWlucyBvbmx5IHRoZSBtb3N0IHJlY2VudCBzZXF1ZW5jZSAocmVjb21tZW5kZWQgZm9yIHZhbGlkYXRpb24vdGVzdCBldmFsdWF0aW9uKS4KICAgICAgICBzdHJpZGU6IFN0cmlkZSBzdGVwIHNpemUgd2hlbiBzbGlkaW5nX3dpbmRvdyBpcyBUcnVlLiBEZWZhdWx0cyB0byBtYXhfbGVuZ3RoIC8vIDIuCiAgICAgICAgcGFkX3Rva2VuX2lkOiBTcGVjaWFsIHRva2VuIGluZGV4IGZvciBwYWRkaW5nIChkZWZhdWx0IDApLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICBoaXN0b3J5X2RmOiBPcHRpb25hbFtwZC5EYXRhRnJhbWVdID0gTm9uZSwKICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgICAgICB1c2VyX2NvbDogc3RyID0gInVzZXJfaWQiLAogICAgICAgIGl0ZW1fdG9rZW5fY29sOiBzdHIgPSAiaXRlbV90b2tlbiIsCiAgICAgICAgdGltZXN0YW1wX2NvbDogc3RyID0gInRpbWVzdGFtcCIsCiAgICAgICAgZHVyYXRpb25fY29sOiBzdHIgPSAibGlzdGVuX2R1cmF0aW9uX3NlYyIsCiAgICAgICAgc2xpZGluZ193aW5kb3c6IGJvb2wgPSBGYWxzZSwKICAgICAgICBzdHJpZGU6IE9wdGlvbmFsW2ludF0gPSBOb25lLAogICAgICAgIHBhZF90b2tlbl9pZDogaW50ID0gMCwKICAgICkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm1heF9sZW5ndGg6IGludCA9IG1heF9sZW5ndGgKICAgICAgICBzZWxmLnBhZF90b2tlbl9pZDogaW50ID0gcGFkX3Rva2VuX2lkCiAgICAgICAgc2VsZi5zbGlkaW5nX3dpbmRvdzogYm9vbCA9IHNsaWRpbmdfd2luZG93CiAgICAgICAgc2VsZi5zdHJpZGU6IGludCA9IHN0cmlkZSBpZiBzdHJpZGUgaXMgbm90IE5vbmUgZWxzZSBtYXgoMSwgbWF4X2xlbmd0aCAvLyAyKQoKICAgICAgICAjIEluIGV2YWx1YXRpb24gKHZhbC90ZXN0KSwgY29uY2F0ZW5hdGUgaGlzdG9yaWNhbCBpbnRlcmFjdGlvbnMgd2l0aCB0YXJnZXQgc3BsaXQKICAgICAgICAjIHRvIGVuc3VyZSB1c2VycyBoYXZlIGZ1bGwgY2hyb25vbG9naWNhbCBjb250ZXh0IHVwIHRvIHRoZSBldmFsdWF0aW9uIHRhcmdldC4KICAgICAgICBpZiBoaXN0b3J5X2RmIGlzIG5vdCBOb25lOgogICAgICAgICAgICAjIEZpbHRlciBoaXN0b3J5IHRvIG9ubHkgdXNlcnMgcHJlc2VudCBpbiBjdXJyZW50IHRhcmdldCBzcGxpdAogICAgICAgICAgICBhY3RpdmVfdXNlcnMgPSBzZXQoZGZbdXNlcl9jb2xdLnVuaXF1ZSgpKQogICAgICAgICAgICBmaWx0ZXJlZF9oaXN0b3J5ID0gaGlzdG9yeV9kZltoaXN0b3J5X2RmW3VzZXJfY29sXS5pc2luKGFjdGl2ZV91c2VycyldCiAgICAgICAgICAgIGNvbWJpbmVkX2RmID0gcGQuY29uY2F0KFtmaWx0ZXJlZF9oaXN0b3J5LCBkZl0sIGlnbm9yZV9pbmRleD1UcnVlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNvbWJpbmVkX2RmID0gZGYKCiAgICAgICAgIyBTdG9yZSBzZXF1ZW5jZSBzYW1wbGVzOiBMaXN0IG9mIHR1cGxlcyAodXNlcl9pZCwgaXRlbV9zZXEsIGR1cmF0aW9uX3NlcSkKICAgICAgICBzZWxmLnNhbXBsZXM6IExpc3RbVHVwbGVbVW5pb25bc3RyLCBpbnRdLCBucC5uZGFycmF5LCBucC5uZGFycmF5XV0gPSBbXQogICAgICAgIHNlbGYuX2J1aWxkX3NlcXVlbmNlcygKICAgICAgICAgICAgZGY9Y29tYmluZWRfZGYsCiAgICAgICAgICAgIHVzZXJfY29sPXVzZXJfY29sLAogICAgICAgICAgICBpdGVtX3Rva2VuX2NvbD1pdGVtX3Rva2VuX2NvbCwKICAgICAgICAgICAgdGltZXN0YW1wX2NvbD10aW1lc3RhbXBfY29sLAogICAgICAgICAgICBkdXJhdGlvbl9jb2w9ZHVyYXRpb25fY29sLAogICAgICAgICkKCiAgICBkZWYgX2J1aWxkX3NlcXVlbmNlcygKICAgICAgICBzZWxmLAogICAgICAgIGRmOiBwZC5EYXRhRnJhbWUsCiAgICAgICAgdXNlcl9jb2w6IHN0ciwKICAgICAgICBpdGVtX3Rva2VuX2NvbDogc3RyLAogICAgICAgIHRpbWVzdGFtcF9jb2w6IHN0ciwKICAgICAgICBkdXJhdGlvbl9jb2w6IHN0ciwKICAgICkgLT4gTm9uZToKICAgICAgICAiIiJHcm91cHMgdXNlciBpbnRlcmFjdGlvbnMsIHNvcnRzIGNocm9ub2xvZ2ljYWxseSwgYW5kIGNvbnN0cnVjdHMgc2VxdWVuY2VzLiIiIgogICAgICAgICMgRW5zdXJlIGNocm9ub2xvZ2ljYWwgb3JkZXJpbmcKICAgICAgICBzb3J0ZWRfZGYgPSBkZi5zb3J0X3ZhbHVlcyhieT1bdXNlcl9jb2wsIHRpbWVzdGFtcF9jb2xdKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICAgICAgaGFzX2R1cmF0aW9uID0gZHVyYXRpb25fY29sIGluIHNvcnRlZF9kZi5jb2x1bW5zCgogICAgICAgIGdyb3VwZWQgPSBzb3J0ZWRfZGYuZ3JvdXBieSh1c2VyX2NvbCkKICAgICAgICBza2lwcGVkX3Nob3J0ID0gMAoKICAgICAgICBmb3IgdXNlcl9pZCwgZ3JvdXAgaW4gZ3JvdXBlZDoKICAgICAgICAgICAgaXRlbV9zZXEgPSBncm91cFtpdGVtX3Rva2VuX2NvbF0udmFsdWVzLmFzdHlwZShucC5pbnQ2NCkKICAgICAgICAgICAgZHVyYXRpb25fc2VxID0gKAogICAgICAgICAgICAgICAgZ3JvdXBbZHVyYXRpb25fY29sXS52YWx1ZXMuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgICAgICAgICBpZiBoYXNfZHVyYXRpb24KICAgICAgICAgICAgICAgIGVsc2UgbnAub25lcyhsZW4oaXRlbV9zZXEpLCBkdHlwZT1ucC5mbG9hdDMyKSAqIDMwLjAKICAgICAgICAgICAgKQoKICAgICAgICAgICAgIyBDYXVzYWwgTE0gbmV4dC1pdGVtIHByZWRpY3Rpb24gcmVxdWlyZXMgYXQgbGVhc3QgMiB0b2tlbnMgKDEgaW5wdXQsIDEgbGFiZWwpCiAgICAgICAgICAgIGlmIGxlbihpdGVtX3NlcSkgPCAyOgogICAgICAgICAgICAgICAgc2tpcHBlZF9zaG9ydCArPSAxCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgaWYgc2VsZi5zbGlkaW5nX3dpbmRvdzoKICAgICAgICAgICAgICAgICMgU2xpZGluZyB3aW5kb3cgY2h1bmtpbmcgZm9yIHRyYWluaW5nIGRhdGEgYXVnbWVudGF0aW9uCiAgICAgICAgICAgICAgICAjIEEgd2luZG93IG9mIHNpemUgKG1heF9sZW5ndGggKyAxKSBwcm92aWRlcyBtYXhfbGVuZ3RoIGlucHV0cyBhbmQgbWF4X2xlbmd0aCBzaGlmdGVkIGxhYmVscwogICAgICAgICAgICAgICAgd2luZG93X3NpemUgPSBzZWxmLm1heF9sZW5ndGggKyAxCiAgICAgICAgICAgICAgICBpZiBsZW4oaXRlbV9zZXEpIDw9IHdpbmRvd19zaXplOgogICAgICAgICAgICAgICAgICAgIHNlbGYuc2FtcGxlcy5hcHBlbmQoKHVzZXJfaWQsIGl0ZW1fc2VxLCBkdXJhdGlvbl9zZXEpKQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBmb3Igc3RhcnRfaWR4IGluIHJhbmdlKDAsIGxlbihpdGVtX3NlcSkgLSAxLCBzZWxmLnN0cmlkZSk6CiAgICAgICAgICAgICAgICAgICAgICAgIGVuZF9pZHggPSBtaW4oc3RhcnRfaWR4ICsgd2luZG93X3NpemUsIGxlbihpdGVtX3NlcSkpCiAgICAgICAgICAgICAgICAgICAgICAgIGNodW5rX2l0ZW1zID0gaXRlbV9zZXFbc3RhcnRfaWR4OmVuZF9pZHhdCiAgICAgICAgICAgICAgICAgICAgICAgIGNodW5rX2R1cmF0aW9ucyA9IGR1cmF0aW9uX3NlcVtzdGFydF9pZHg6ZW5kX2lkeF0KICAgICAgICAgICAgICAgICAgICAgICAgaWYgbGVuKGNodW5rX2l0ZW1zKSA+PSAyOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5zYW1wbGVzLmFwcGVuZCgodXNlcl9pZCwgY2h1bmtfaXRlbXMsIGNodW5rX2R1cmF0aW9ucykpCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGVuZF9pZHggPT0gbGVuKGl0ZW1fc2VxKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAjIEV2YWx1YXRpb24gbW9kZTogcmV0YWluIHRoZSBtb3N0IHJlY2VudCBoaXN0b3J5IHVwIHRvIChtYXhfbGVuZ3RoICsgMSkKICAgICAgICAgICAgICAgIGNodW5rX2l0ZW1zID0gaXRlbV9zZXFbLShzZWxmLm1heF9sZW5ndGggKyAxKTpdCiAgICAgICAgICAgICAgICBjaHVua19kdXJhdGlvbnMgPSBkdXJhdGlvbl9zZXFbLShzZWxmLm1heF9sZW5ndGggKyAxKTpdCiAgICAgICAgICAgICAgICBzZWxmLnNhbXBsZXMuYXBwZW5kKCh1c2VyX2lkLCBjaHVua19pdGVtcywgY2h1bmtfZHVyYXRpb25zKSkKCiAgICAgICAgbG9nZ2VyLmluZm8oCiAgICAgICAgICAgICJDb25zdHJ1Y3RlZCAlZCBzZXF1ZW5jZSBzYW1wbGVzIGZyb20gJWQgdXNlcnMgKHNraXBwZWQgJWQgdXNlcnMgd2l0aCA8MiBpbnRlcmFjdGlvbnMpLiAiCiAgICAgICAgICAgICJzbGlkaW5nX3dpbmRvdz0lcywgbWF4X2xlbmd0aD0lZCIsCiAgICAgICAgICAgIGxlbihzZWxmLnNhbXBsZXMpLAogICAgICAgICAgICBsZW4oZ3JvdXBlZCksCiAgICAgICAgICAgIHNraXBwZWRfc2hvcnQsCiAgICAgICAgICAgIHNlbGYuc2xpZGluZ193aW5kb3csCiAgICAgICAgICAgIHNlbGYubWF4X2xlbmd0aCwKICAgICAgICApCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5zYW1wbGVzKQoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpZHg6IGludCkgLT4gRGljdFtzdHIsIFVuaW9uW25wLm5kYXJyYXksIFVuaW9uW3N0ciwgaW50XV1dOgogICAgICAgICIiIgogICAgICAgIFJldHJpZXZlcyBhIHNpbmdsZSBzZXF1ZW5jZSBzYW1wbGUsIGFsaWducyBpbnB1dHMgYW5kIHNoaWZ0ZWQgY2F1c2FsIGxhYmVscywKICAgICAgICBhbmQgYXBwbGllcyB6ZXJvLXBhZGRpbmcgdG8gcmVhY2ggbWF4X2xlbmd0aC4KCiAgICAgICAgTGV0IHJhdyBpdGVtcyBiZSBbc18xLCBzXzIsIC4uLiwgc19rLCBzX3trKzF9XS4KICAgICAgICAtIElucHV0IHNlcXVlbmNlOiAgIFtzXzEsIHNfMiwgLi4uLCBzX2tdCiAgICAgICAgLSBTaGlmdGVkIExhYmVsOiAgICBbc18yLCBzXzMsIC4uLiwgc197aysxfV0KICAgICAgICAtIFNoaWZ0ZWQgRHVyYXRpb246IFtyXzIsIHJfMywgLi4uLCByX3trKzF9XQogICAgICAgICIiIgogICAgICAgIHVzZXJfaWQsIGl0ZW1zLCBkdXJhdGlvbnMgPSBzZWxmLnNhbXBsZXNbaWR4XQoKICAgICAgICAjIFNoaWZ0IHNlcXVlbmNlIGZvciBuZXh0LXRva2VuIHByZWRpY3Rpb24KICAgICAgICByYXdfaW5wdXRzID0gaXRlbXNbOi0xXQogICAgICAgIHJhd19sYWJlbHMgPSBpdGVtc1sxOl0KICAgICAgICByYXdfZHVyYXRpb25zID0gZHVyYXRpb25zWzE6XQoKICAgICAgICAjIFRydW5jYXRlIGlmIGxvbmdlciB0aGFuIG1heF9sZW5ndGggKGUuZy4gZnJvbSByaWdodG1vc3QgcmVjZW50IGhpc3RvcnkpCiAgICAgICAgaWYgbGVuKHJhd19pbnB1dHMpID4gc2VsZi5tYXhfbGVuZ3RoOgogICAgICAgICAgICByYXdfaW5wdXRzID0gcmF3X2lucHV0c1stc2VsZi5tYXhfbGVuZ3RoOl0KICAgICAgICAgICAgcmF3X2xhYmVscyA9IHJhd19sYWJlbHNbLXNlbGYubWF4X2xlbmd0aDpdCiAgICAgICAgICAgIHJhd19kdXJhdGlvbnMgPSByYXdfZHVyYXRpb25zWy1zZWxmLm1heF9sZW5ndGg6XQoKICAgICAgICBjdXJyX2xlbiA9IGxlbihyYXdfaW5wdXRzKQogICAgICAgIHBhZF9sZW4gPSBzZWxmLm1heF9sZW5ndGggLSBjdXJyX2xlbgoKICAgICAgICAjIFJpZ2h0LXBhZGRpbmcgd2l0aCBwYWRfdG9rZW5faWQgKDApCiAgICAgICAgIyBJbnB1dCBJRHM6IFt0b2tlbl8xLCB0b2tlbl8yLCAuLi4sIHRva2VuX2ssIDAsIC4uLiwgMF0KICAgICAgICBpbnB1dF9pZHMgPSBucC5wYWQocmF3X2lucHV0cywgKDAsIHBhZF9sZW4pLCBjb25zdGFudF92YWx1ZXM9c2VsZi5wYWRfdG9rZW5faWQpCiAgICAgICAgIyBMYWJlbHM6IFt0b2tlbl8yLCB0b2tlbl8zLCAuLi4sIHRva2VuX3trKzF9LCAwLCAuLi4sIDBdCiAgICAgICAgbGFiZWxzID0gbnAucGFkKHJhd19sYWJlbHMsICgwLCBwYWRfbGVuKSwgY29uc3RhbnRfdmFsdWVzPXNlbGYucGFkX3Rva2VuX2lkKQogICAgICAgICMgQXR0ZW50aW9uIE1hc2s6IDEgZm9yIGFjdGl2ZSBpdGVtIHRva2VucywgMCBmb3IgPFBBRD4KICAgICAgICBhdHRlbnRpb25fbWFzayA9IG5wLnplcm9zKHNlbGYubWF4X2xlbmd0aCwgZHR5cGU9bnAuaW50NjQpCiAgICAgICAgYXR0ZW50aW9uX21hc2tbOmN1cnJfbGVuXSA9IDEKICAgICAgICAjIER1cmF0aW9uczogYWxpZ25lZCB3aXRoIGxhYmVscyBmb3IgUkwgcmV3YXJkcwogICAgICAgIGR1cmF0aW9uc19wYWRkZWQgPSBucC5wYWQocmF3X2R1cmF0aW9ucywgKDAsIHBhZF9sZW4pLCBjb25zdGFudF92YWx1ZXM9MC4wKQoKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiaW5wdXRfaWRzIjogaW5wdXRfaWRzLAogICAgICAgICAgICAibGFiZWxzIjogbGFiZWxzLAogICAgICAgICAgICAiYXR0ZW50aW9uX21hc2siOiBhdHRlbnRpb25fbWFzaywKICAgICAgICAgICAgImR1cmF0aW9ucyI6IGR1cmF0aW9uc19wYWRkZWQsCiAgICAgICAgICAgICJ1c2VyX2lkIjogdXNlcl9pZCwKICAgICAgICB9CgoKZGVmIHJlY3N5c19jb2xsYXRlX2ZuKGJhdGNoX2xpc3Q6IExpc3RbRGljdFtzdHIsIFVuaW9uW25wLm5kYXJyYXksIFVuaW9uW3N0ciwgaW50XV1dXSkgLT4gUmVjQmF0Y2g6CiAgICAiIiIKICAgIENvbGxhdGUgZnVuY3Rpb24gdGhhdCBiYXRjaGVzIGluZGl2aWR1YWwgZGljdGlvbmFyeSBzYW1wbGVzIGludG8gYSB1bmlmaWVkIFJlY0JhdGNoLgogICAgIiIiCiAgICBpbnB1dF9pZHMgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFtiWyJpbnB1dF9pZHMiXSBmb3IgYiBpbiBiYXRjaF9saXN0XSkpLmxvbmcoKQogICAgbGFiZWxzID0gdG9yY2guZnJvbV9udW1weShucC5zdGFjayhbYlsibGFiZWxzIl0gZm9yIGIgaW4gYmF0Y2hfbGlzdF0pKS5sb25nKCkKICAgIGF0dGVudGlvbl9tYXNrID0gdG9yY2guZnJvbV9udW1weShucC5zdGFjayhbYlsiYXR0ZW50aW9uX21hc2siXSBmb3IgYiBpbiBiYXRjaF9saXN0XSkpLmxvbmcoKQogICAgZHVyYXRpb25zID0gdG9yY2guZnJvbV9udW1weShucC5zdGFjayhbYlsiZHVyYXRpb25zIl0gZm9yIGIgaW4gYmF0Y2hfbGlzdF0pKS5mbG9hdCgpCiAgICB1c2VyX2lkcyA9IFtiWyJ1c2VyX2lkIl0gZm9yIGIgaW4gYmF0Y2hfbGlzdF0KCiAgICByZXR1cm4gUmVjQmF0Y2goCiAgICAgICAgaW5wdXRfaWRzPWlucHV0X2lkcywKICAgICAgICBsYWJlbHM9bGFiZWxzLAogICAgICAgIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrLAogICAgICAgIGR1cmF0aW9ucz1kdXJhdGlvbnMsCiAgICAgICAgdXNlcl9pZHM9dXNlcl9pZHMsCiAgICApCgoKZGVmIGNyZWF0ZV9kYXRhbG9hZGVyKAogICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgIGhpc3RvcnlfZGY6IE9wdGlvbmFsW3BkLkRhdGFGcmFtZV0gPSBOb25lLAogICAgbWF4X2xlbmd0aDogaW50ID0gNjQsCiAgICBiYXRjaF9zaXplOiBpbnQgPSAxMjgsCiAgICBzaHVmZmxlOiBib29sID0gVHJ1ZSwKICAgIHNsaWRpbmdfd2luZG93OiBib29sID0gRmFsc2UsCiAgICBzdHJpZGU6IE9wdGlvbmFsW2ludF0gPSBOb25lLAogICAgbnVtX3dvcmtlcnM6IGludCA9IDAsCiAgICBwaW5fbWVtb3J5OiBib29sID0gRmFsc2UsCiAgICBwYWRfdG9rZW5faWQ6IGludCA9IDAsCikgLT4gRGF0YUxvYWRlcjoKICAgICIiIgogICAgRmFjdG9yeSBmdW5jdGlvbiBjcmVhdGluZyBhbiBvcHRpbWl6ZWQgUHlUb3JjaCBEYXRhTG9hZGVyIGZvciBzZXF1ZW50aWFsIHJlY29tbWVuZGF0aW9uLgogICAgIiIiCiAgICBkYXRhc2V0ID0gU2VxdWVudGlhbFJlY0RhdGFzZXQoCiAgICAgICAgZGY9ZGYsCiAgICAgICAgaGlzdG9yeV9kZj1oaXN0b3J5X2RmLAogICAgICAgIG1heF9sZW5ndGg9bWF4X2xlbmd0aCwKICAgICAgICBzbGlkaW5nX3dpbmRvdz1zbGlkaW5nX3dpbmRvdywKICAgICAgICBzdHJpZGU9c3RyaWRlLAogICAgICAgIHBhZF90b2tlbl9pZD1wYWRfdG9rZW5faWQsCiAgICApCgogICAgcmV0dXJuIERhdGFMb2FkZXIoCiAgICAgICAgZGF0YXNldD1kYXRhc2V0LAogICAgICAgIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwKICAgICAgICBzaHVmZmxlPXNodWZmbGUsCiAgICAgICAgbnVtX3dvcmtlcnM9bnVtX3dvcmtlcnMsCiAgICAgICAgY29sbGF0ZV9mbj1yZWNzeXNfY29sbGF0ZV9mbiwKICAgICAgICBwaW5fbWVtb3J5PXBpbl9tZW1vcnksCiAgICAgICAgZHJvcF9sYXN0PUZhbHNlLAogICAgKQo=",
    "model.py": "IiIiCm1vZGVsLnB5Ci0tLS0tLS0tCkNhdXNhbCBMYW5ndWFnZSBNb2RlbCAoR1BULTIpIGZvciBBdXRvcmVncmVzc2l2ZSBOZXh0LUl0ZW0gUmVjb21tZW5kYXRpb24uCgpJbml0aWFsaXplcyBhIGNvbXBhY3QgR1BULTIgYXJjaGl0ZWN0dXJlIGZyb20gc2NyYXRjaCAobm8gcHJlLXRyYWluZWQgTkxQIHdlaWdodHMpCnRhaWxvcmVkIGZvciBpdGVtIGNhdGFsb2dzIGFuZCBzZXF1ZW50aWFsIGludGVyYWN0aW9uIGR5bmFtaWNzLgoKQXJjaGl0ZWN0dXJlIFNwZWNpZmljYXRpb25zOgogICAgLSBCYXNlIENsYXNzOiB0cmFuc2Zvcm1lcnMuR1BUMkxNSGVhZE1vZGVsIChpbml0aWFsaXplZCBmcm9tIHNjcmF0Y2gpCiAgICAtIHZvY2FiX3NpemUgPSBudW1faXRlbXMgKyAxIChJbmRleCAwIGlzIDxQQUQ+KQogICAgLSBuX2VtYmQgPSAxMjggKENvbXBhY3QgbGF0ZW50IGVtYmVkZGluZ3MpCiAgICAtIG5fbGF5ZXIgPSAzIChUcmFuc2Zvcm1lciBkZWNvZGVyIGJsb2NrcykKICAgIC0gbl9oZWFkID0gNCAoTXVsdGktaGVhZCBzZWxmLWF0dGVudGlvbiBoZWFkcykKICAgIC0gbl9wb3NpdGlvbnMgPSBtYXhfbGVuZ3RoIChDb250ZXh0IHdpbmRvdyBsZW5ndGgpCiAgICAtIHBhZF90b2tlbl9pZCA9IDAKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIE9wdGlvbmFsLCBUdXBsZSwgVW5pb24KCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEdQVDJDb25maWcsIEdQVDJMTUhlYWRNb2RlbAoKbG9nZ2luZy5iYXNpY0NvbmZpZygKICAgIGxldmVsPWxvZ2dpbmcuSU5GTywKICAgIGZvcm1hdD0iWyUoYXNjdGltZSlzXSBbJShsZXZlbG5hbWUpc10gKCUobmFtZSlzKSAlKG1lc3NhZ2UpcyIsCiAgICBkYXRlZm10PSIlWS0lbS0lZCAlSDolTTolUyIsCikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIk1vZGVsIikKCgpAZGF0YWNsYXNzCmNsYXNzIFJlY1N5c091dHB1dDoKICAgICIiIgogICAgU3RhbmRhcmRpemVkIG1vZGVsIG91dHB1dCBjb250YWluZXIgZm9yIHNlcXVlbnRpYWwgcmVjb21tZW5kYXRpb24uCiAgICAKICAgIEF0dHJpYnV0ZXM6CiAgICAgICAgbG9naXRzOiBGbG9hdFRlbnNvciBbQiwgTCwgdm9jYWJfc2l6ZV0gdW5ub3JtYWxpemVkIHByZWRpY3Rpb24gc2NvcmVzLgogICAgICAgIGxvc3M6IE9wdGlvbmFsIHNjYWxhciBjcm9zcy1lbnRyb3B5IGxvc3MgKGlmIGxhYmVscyB3ZXJlIHByb3ZpZGVkKS4KICAgICAgICBsYXN0X2hpZGRlbl9zdGF0ZTogT3B0aW9uYWwgRmxvYXRUZW5zb3IgW0IsIEwsIG5fZW1iZF0gY29udGV4dHVhbCByZXByZXNlbnRhdGlvbnMuCiAgICAiIiIKICAgIGxvZ2l0czogdG9yY2guVGVuc29yCiAgICBsb3NzOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZQogICAgbGFzdF9oaWRkZW5fc3RhdGU6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lCgoKY2xhc3MgUmVjU3lzR1BUMihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBTZXF1ZW5jZS1iYXNlZCBSZWNvbW1lbmRhdGlvbiBNb2RlbCB1c2luZyBhIENhdXNhbCBHUFQtMiBUcmFuc2Zvcm1lciBpbml0aWFsaXplZCBmcm9tIHNjcmF0Y2guCgogICAgSW4gTmV4dC1JdGVtIFByZWRpY3Rpb24sIGVhY2ggaW50ZXJhY3Rpb24gdG9rZW4gc2VxdWVuY2UgW3NfMSwgLi4uLCBzX2tdIGlzIHByb2plY3RlZAogICAgaW50byBhbiBlbWJlZGRpbmcgc3BhY2UsIHBhc3NlZCB0aHJvdWdoIGNhdXNhbCBzZWxmLWF0dGVudGlvbiBsYXllcnMgd2l0aCBhdXRvcmVncmVzc2l2ZQogICAgbWFza2luZywgYW5kIHByb2plY3RlZCB0byBjYXRhbG9nIGxvZ2l0cyBbQiwgTCwgdm9jYWJfc2l6ZV0uCgogICAgQXJnczoKICAgICAgICBudW1faXRlbXM6IFRvdGFsIG51bWJlciBvZiBhY3RpdmUgaXRlbXMgaW4gY2F0YWxvZyAoZXhjbHVkaW5nIDxQQUQ+KS4KICAgICAgICBtYXhfbGVuZ3RoOiBNYXhpbXVtIHNlcXVlbmNlIGNvbnRleHQgbGVuZ3RoIChuX3Bvc2l0aW9ucykuCiAgICAgICAgbl9lbWJkOiBMYXRlbnQgZW1iZWRkaW5nIGRpbWVuc2lvbiAoZGVmYXVsdDogMTI4KS4KICAgICAgICBuX2xheWVyOiBOdW1iZXIgb2YgdHJhbnNmb3JtZXIgZGVjb2RlciBsYXllcnMgKGRlZmF1bHQ6IDMpLgogICAgICAgIG5faGVhZDogTnVtYmVyIG9mIGF0dGVudGlvbiBoZWFkcyAoZGVmYXVsdDogNCkuCiAgICAgICAgZHJvcG91dDogRHJvcG91dCBwcm9iYWJpbGl0eSBhcHBsaWVkIHRvIGVtYmVkZGluZ3MgYW5kIHJlc2lkdWFsIGNvbm5lY3Rpb25zLgogICAgICAgIHBhZF90b2tlbl9pZDogUmVzZXJ2ZWQgcGFkZGluZyB0b2tlbiBpbmRleCAoZGVmYXVsdDogMCkuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBudW1faXRlbXM6IGludCwKICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgICAgICBuX2VtYmQ6IGludCA9IDEyOCwKICAgICAgICBuX2xheWVyOiBpbnQgPSAzLAogICAgICAgIG5faGVhZDogaW50ID0gNCwKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMSwKICAgICAgICBwYWRfdG9rZW5faWQ6IGludCA9IDAsCiAgICApIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5udW1faXRlbXM6IGludCA9IG51bV9pdGVtcwogICAgICAgIHNlbGYubWF4X2xlbmd0aDogaW50ID0gbWF4X2xlbmd0aAogICAgICAgIHNlbGYucGFkX3Rva2VuX2lkOiBpbnQgPSBwYWRfdG9rZW5faWQKCiAgICAgICAgIyBWb2NhYiBzaXplID0gbnVtX2l0ZW1zICsgMSAoaXRlbSBJRHMgMS4uTiwgcGx1cyAwIGZvciBQQUQpCiAgICAgICAgdm9jYWJfc2l6ZSA9IG51bV9pdGVtcyArIDEKCiAgICAgICAgIyBDb25maWd1cmUgY3VzdG9tIEdQVC0yIGFyY2hpdGVjdHVyZQogICAgICAgIHNlbGYuY29uZmlnID0gR1BUMkNvbmZpZygKICAgICAgICAgICAgdm9jYWJfc2l6ZT12b2NhYl9zaXplLAogICAgICAgICAgICBuX3Bvc2l0aW9ucz1tYXhfbGVuZ3RoLAogICAgICAgICAgICBuX2N0eD1tYXhfbGVuZ3RoLAogICAgICAgICAgICBuX2VtYmQ9bl9lbWJkLAogICAgICAgICAgICBuX2xheWVyPW5fbGF5ZXIsCiAgICAgICAgICAgIG5faGVhZD1uX2hlYWQsCiAgICAgICAgICAgIHJlc2lkX3Bkcm9wPWRyb3BvdXQsCiAgICAgICAgICAgIGVtYmRfcGRyb3A9ZHJvcG91dCwKICAgICAgICAgICAgYXR0bl9wZHJvcD1kcm9wb3V0LAogICAgICAgICAgICBwYWRfdG9rZW5faWQ9cGFkX3Rva2VuX2lkLAogICAgICAgICAgICBib3NfdG9rZW5faWQ9Tm9uZSwgICMgQ2xlYXIgZGVmYXVsdCB0ZXh0IE5MUCB0b2tlbnMKICAgICAgICAgICAgZW9zX3Rva2VuX2lkPU5vbmUsCiAgICAgICAgKQoKICAgICAgICAjIFNUUklDVCBSRVFVSVJFTUVOVDogSW5pdGlhbGl6ZWQgZnJvbSBzY3JhdGNoIChOT1QgZnJvbSBwcmUtdHJhaW5lZCB3ZWlnaHRzKQogICAgICAgIHNlbGYudHJhbnNmb3JtZXIgPSBHUFQyTE1IZWFkTW9kZWwoc2VsZi5jb25maWcpCgogICAgICAgICMgQ3Jvc3MtRW50cm9weSBMb3NzIHdpdGggaWdub3JlX2luZGV4PTAgKDxQQUQ+KQogICAgICAgIHNlbGYubG9zc19mY3QgPSBubi5Dcm9zc0VudHJvcHlMb3NzKGlnbm9yZV9pbmRleD1zZWxmLnBhZF90b2tlbl9pZCkKCiAgICAgICAgIyBDb3VudCBwYXJhbWV0ZXJzCiAgICAgICAgbnVtX3BhcmFtcyA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkpCiAgICAgICAgbnVtX3RyYWluYWJsZSA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKQogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICAiSW5pdGlhbGl6ZWQgUmVjU3lzR1BUMiBmcm9tIHNjcmF0Y2g6IHZvY2FiX3NpemU9JWQgKG51bV9pdGVtcz0lZCArIFBBRCksICIKICAgICAgICAgICAgIm5fZW1iZD0lZCwgbl9sYXllcj0lZCwgbl9oZWFkPSVkLCBtYXhfbGVuZ3RoPSVkLiAiCiAgICAgICAgICAgICJUb3RhbCBwYXJhbWV0ZXJzOiAlZCAodHJhaW5hYmxlOiAlZCkiLAogICAgICAgICAgICB2b2NhYl9zaXplLAogICAgICAgICAgICBudW1faXRlbXMsCiAgICAgICAgICAgIG5fZW1iZCwKICAgICAgICAgICAgbl9sYXllciwKICAgICAgICAgICAgbl9oZWFkLAogICAgICAgICAgICBtYXhfbGVuZ3RoLAogICAgICAgICAgICBudW1fcGFyYW1zLAogICAgICAgICAgICBudW1fdHJhaW5hYmxlLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIGlucHV0X2lkczogdG9yY2guVGVuc29yLAogICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZSwKICAgICAgICBsYWJlbHM6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgKSAtPiBSZWNTeXNPdXRwdXQ6CiAgICAgICAgIiIiCiAgICAgICAgRm9yd2FyZCBwYXNzIHRocm91Z2ggQ2F1c2FsIEdQVC0yLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgaXRlbSB0b2tlbnMuCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbCBMb25nVGVuc29yIFtCLCBMXSB3aXRoIDEgZm9yIHJlYWwgaXRlbXMsIDAgZm9yIDxQQUQ+LgogICAgICAgICAgICBsYWJlbHM6IE9wdGlvbmFsIExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgc2hpZnRlZCBncm91bmQtdHJ1dGggaXRlbSB0b2tlbnMuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIFJlY1N5c091dHB1dCBjb250YWluaW5nIGxvZ2l0cyBbQiwgTCwgdm9jYWJfc2l6ZV0sIG9wdGlvbmFsIHNjYWxhciBsb3NzLAogICAgICAgICAgICBhbmQgb3B0aW9uYWwgbGFzdCBoaWRkZW4gc3RhdGUgcmVwcmVzZW50YXRpb25zIFtCLCBMLCBuX2VtYmRdLgogICAgICAgICIiIgogICAgICAgICMgQXV0by1nZW5lcmF0ZSBhdHRlbnRpb24gbWFzayBpZiBvbWl0dGVkCiAgICAgICAgaWYgYXR0ZW50aW9uX21hc2sgaXMgTm9uZToKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSAoaW5wdXRfaWRzICE9IHNlbGYucGFkX3Rva2VuX2lkKS5sb25nKCkKCiAgICAgICAgIyBGb3J3YXJkIHRocm91Z2ggdHJhbnNmb3JtZXIgKHdpdGhvdXQgaW50ZXJuYWwgbGFiZWxzIHRvIGNvbXB1dGUgY3VzdG9tIGxvc3MpCiAgICAgICAgdHJhbnNmb3JtZXJfb3V0cHV0cyA9IHNlbGYudHJhbnNmb3JtZXIoCiAgICAgICAgICAgIGlucHV0X2lkcz1pbnB1dF9pZHMsCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrLAogICAgICAgICAgICBvdXRwdXRfaGlkZGVuX3N0YXRlcz1UcnVlLAogICAgICAgICAgICByZXR1cm5fZGljdD1UcnVlLAogICAgICAgICkKCiAgICAgICAgbG9naXRzID0gdHJhbnNmb3JtZXJfb3V0cHV0cy5sb2dpdHMgICMgW0IsIEwsIHZvY2FiX3NpemVdCiAgICAgICAgbGFzdF9oaWRkZW5fc3RhdGUgPSB0cmFuc2Zvcm1lcl9vdXRwdXRzLmhpZGRlbl9zdGF0ZXNbLTFdICAjIFtCLCBMLCBuX2VtYmRdCgogICAgICAgIGxvc3MgPSBOb25lCiAgICAgICAgaWYgbGFiZWxzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAjIEZsYXR0ZW4gYmF0Y2ggYW5kIHNlcXVlbmNlIGRpbWVuc2lvbnMgZm9yIGNyb3NzLWVudHJvcHkKICAgICAgICAgICAgIyBQYWRkZWQgcG9zaXRpb25zIGluIGxhYmVscyAodmFsdWUgMCkgYXJlIGlnbm9yZWQgaW4gbG9zcyBjb21wdXRhdGlvbgogICAgICAgICAgICBsb3NzID0gc2VsZi5sb3NzX2ZjdCgKICAgICAgICAgICAgICAgIGxvZ2l0cy52aWV3KC0xLCBsb2dpdHMuc2l6ZSgtMSkpLAogICAgICAgICAgICAgICAgbGFiZWxzLnZpZXcoLTEpLAogICAgICAgICAgICApCgogICAgICAgIHJldHVybiBSZWNTeXNPdXRwdXQoCiAgICAgICAgICAgIGxvZ2l0cz1sb2dpdHMsCiAgICAgICAgICAgIGxvc3M9bG9zcywKICAgICAgICAgICAgbGFzdF9oaWRkZW5fc3RhdGU9bGFzdF9oaWRkZW5fc3RhdGUsCiAgICAgICAgKQoKICAgIGRlZiBnZXRfbGFzdF9hY3Rpb25fc3RhdGVzKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgRXh0cmFjdHMgdGhlIGNvbnRleHR1YWwgcmVwcmVzZW50YXRpb24gdmVjdG9yIGhfdCBhdCB0aGUgbGFzdCB2YWxpZCBpbnRlcmFjdGlvbiB0b2tlbi4KICAgICAgICAKICAgICAgICBFc3NlbnRpYWwgZm9yIFBoYXNlIDIgKFJMIEZpbmUtVHVuaW5nKTogVGhpcyB2ZWN0b3Igc2VydmVzIGFzIHRoZSBNRFAgU3RhdGUgc190CiAgICAgICAgZm9yIHRoZSBSTCBwb2xpY3kgYWN0b3IvY3JpdGljIG5ldHdvcmtzLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbCBMb25nVGVuc29yIFtCLCBMXQoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBGbG9hdFRlbnNvciBbQiwgbl9lbWJkXSBzdGF0ZSByZXByZXNlbnRhdGlvbnMuCiAgICAgICAgIiIiCiAgICAgICAgaWYgYXR0ZW50aW9uX21hc2sgaXMgTm9uZToKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSAoaW5wdXRfaWRzICE9IHNlbGYucGFkX3Rva2VuX2lkKS5sb25nKCkKCiAgICAgICAgb3V0ID0gc2VsZi5mb3J3YXJkKGlucHV0X2lkcz1pbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrKQogICAgICAgIGhpZGRlbiA9IG91dC5sYXN0X2hpZGRlbl9zdGF0ZSAgIyBbQiwgTCwgbl9lbWJkXQoKICAgICAgICAjIEZpbmQgdGhlIGluZGV4IG9mIHRoZSBsYXN0IGFjdGl2ZSB0b2tlbiBpbiBlYWNoIHNlcXVlbmNlCiAgICAgICAgIyBTZXF1ZW5jZSBsZW5ndGhzOiBzdW0gYWNyb3NzIGF0dGVudGlvbl9tYXNrCiAgICAgICAgc2VxX2xlbnMgPSBhdHRlbnRpb25fbWFzay5zdW0oZGltPS0xKSAtIDEgICMgMC1pbmRleGVkIFtCXQogICAgICAgIHNlcV9sZW5zID0gdG9yY2guY2xhbXAoc2VxX2xlbnMsIG1pbj0wKQoKICAgICAgICBiYXRjaF9pbmRpY2VzID0gdG9yY2guYXJhbmdlKGlucHV0X2lkcy5zaXplKDApLCBkZXZpY2U9aW5wdXRfaWRzLmRldmljZSkKICAgICAgICBsYXN0X3N0YXRlcyA9IGhpZGRlbltiYXRjaF9pbmRpY2VzLCBzZXFfbGVuc10gICMgW0IsIG5fZW1iZF0KICAgICAgICByZXR1cm4gbGFzdF9zdGF0ZXMKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgcHJlZGljdF9uZXh0X2l0ZW1zKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgICAgIHRvcF9rOiBpbnQgPSAxMCwKICAgICAgICBmaWx0ZXJfaGlzdG9yeTogYm9vbCA9IFRydWUsCiAgICApIC0+IFR1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yXToKICAgICAgICAiIiIKICAgICAgICBHZW5lcmF0ZXMgVG9wLUsgbmV4dCBpdGVtIHJlY29tbWVuZGF0aW9ucyBmb3IgdGhlIGFjdGl2ZSB1c2VyIHNlcXVlbmNlLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbCBMb25nVGVuc29yIFtCLCBMXQogICAgICAgICAgICB0b3BfazogTnVtYmVyIG9mIHJlY29tbWVuZGF0aW9ucyB0byByZXR1cm4gcGVyIHNlcXVlbmNlLgogICAgICAgICAgICBmaWx0ZXJfaGlzdG9yeTogSWYgVHJ1ZSwgbWFza3MgcHJldmlvdXNseSBpbnRlcmFjdGVkIGl0ZW1zIHNvIHRoZXkgYXJlIG5vdCByZS1yZWNvbW1lbmRlZC4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgVHVwbGUgb2YgKHRvcGtfc2NvcmVzIFtCLCB0b3Bfa10sIHRvcGtfaW5kaWNlcyBbQiwgdG9wX2tdKS4KICAgICAgICAiIiIKICAgICAgICBzZWxmLmV2YWwoKQogICAgICAgIGlmIGF0dGVudGlvbl9tYXNrIGlzIE5vbmU6CiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrID0gKGlucHV0X2lkcyAhPSBzZWxmLnBhZF90b2tlbl9pZCkubG9uZygpCgogICAgICAgIG91dCA9IHNlbGYuZm9yd2FyZChpbnB1dF9pZHM9aW5wdXRfaWRzLCBhdHRlbnRpb25fbWFzaz1hdHRlbnRpb25fbWFzaykKICAgICAgICBsb2dpdHMgPSBvdXQubG9naXRzICAjIFtCLCBMLCB2b2NhYl9zaXplXQoKICAgICAgICAjIEV4dHJhY3QgbG9naXRzIGNvcnJlc3BvbmRpbmcgdG8gdGhlIGxhc3Qgb2JzZXJ2ZWQgaXRlbQogICAgICAgIHNlcV9sZW5zID0gYXR0ZW50aW9uX21hc2suc3VtKGRpbT0tMSkgLSAxCiAgICAgICAgc2VxX2xlbnMgPSB0b3JjaC5jbGFtcChzZXFfbGVucywgbWluPTApCiAgICAgICAgYmF0Y2hfaW5kaWNlcyA9IHRvcmNoLmFyYW5nZShpbnB1dF9pZHMuc2l6ZSgwKSwgZGV2aWNlPWlucHV0X2lkcy5kZXZpY2UpCiAgICAgICAgbmV4dF9pdGVtX2xvZ2l0cyA9IGxvZ2l0c1tiYXRjaF9pbmRpY2VzLCBzZXFfbGVuc10gICMgW0IsIHZvY2FiX3NpemVdCgogICAgICAgICMgTmV2ZXIgcmVjb21tZW5kIHRoZSA8UEFEPiB0b2tlbiAoaW5kZXggMCkKICAgICAgICBuZXh0X2l0ZW1fbG9naXRzWzosIHNlbGYucGFkX3Rva2VuX2lkXSA9IC1mbG9hdCgiaW5mIikKCiAgICAgICAgaWYgZmlsdGVyX2hpc3Rvcnk6CiAgICAgICAgICAgICMgTWFzayBvdXQgaXRlbXMgdGhhdCBhcHBlYXIgaW4gdGhlIHVzZXIncyBoaXN0b3JpY2FsIGNvbnRleHQKICAgICAgICAgICAgaGlzdG9yeV9tYXNrID0gdG9yY2guemVyb3NfbGlrZShuZXh0X2l0ZW1fbG9naXRzLCBkdHlwZT10b3JjaC5ib29sKQogICAgICAgICAgICBoaXN0b3J5X21hc2suc2NhdHRlcl8oMSwgaW5wdXRfaWRzLCBUcnVlKQogICAgICAgICAgICBoaXN0b3J5X21hc2tbOiwgc2VsZi5wYWRfdG9rZW5faWRdID0gRmFsc2UgICMgcHJlc2VydmUgcGFkIGZsYWcKICAgICAgICAgICAgbmV4dF9pdGVtX2xvZ2l0cyA9IG5leHRfaXRlbV9sb2dpdHMubWFza2VkX2ZpbGwoaGlzdG9yeV9tYXNrLCAtZmxvYXQoImluZiIpKQoKICAgICAgICAjIENvbXB1dGUgdG9wX2sgc2NvcmVzIGFuZCBpbmRpY2VzCiAgICAgICAgdG9wa19zY29yZXMsIHRvcGtfaW5kaWNlcyA9IHRvcmNoLnRvcGsobmV4dF9pdGVtX2xvZ2l0cywgaz10b3BfaywgZGltPS0xKQogICAgICAgIHJldHVybiB0b3BrX3Njb3JlcywgdG9wa19pbmRpY2VzCgoKZGVmIGJ1aWxkX2dwdDJfcmVjc3lzX21vZGVsKAogICAgbnVtX2l0ZW1zOiBpbnQsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgIG5fZW1iZDogaW50ID0gMTI4LAogICAgbl9sYXllcjogaW50ID0gMywKICAgIG5faGVhZDogaW50ID0gNCwKICAgIGRyb3BvdXQ6IGZsb2F0ID0gMC4xLAogICAgcGFkX3Rva2VuX2lkOiBpbnQgPSAwLAopIC0+IFJlY1N5c0dQVDI6CiAgICAiIiJGYWN0b3J5IGhlbHBlciB0byBidWlsZCBhbmQgcmV0dXJuIGEgUmVjU3lzR1BUMiBtb2RlbCBpbnN0YW5jZS4iIiIKICAgIHJldHVybiBSZWNTeXNHUFQyKAogICAgICAgIG51bV9pdGVtcz1udW1faXRlbXMsCiAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLAogICAgICAgIG5fZW1iZD1uX2VtYmQsCiAgICAgICAgbl9sYXllcj1uX2xheWVyLAogICAgICAgIG5faGVhZD1uX2hlYWQsCiAgICAgICAgZHJvcG91dD1kcm9wb3V0LAogICAgICAgIHBhZF90b2tlbl9pZD1wYWRfdG9rZW5faWQsCiAgICApCg==",
    "train_base.py": "...",
    "requirements.txt": "dG9yY2g+PTIuMC4wCnRyYW5zZm9ybWVycz49NC4zNi4wCnBhbmRhcz49Mi4wLjAKbnVtcHk+PTEuMjIuMApweWFycm93Pj0xNC4wLjAKc2Npa2l0LWxlYXJuPj0xLjMuMAp0cWRtPj00LjY1LjAKd2FuZGI+PTAuMTYuMAo=",
}

for filename, encoded_content in EMBEDDED_MODULES.items():
    if not os.path.exists(filename):
        with open(filename, "wb") as f:
            f.write(base64.b64decode(encoded_content))
        print(f"[Workspace] Auto-created module: {filename}")
    else:
        print(f"[Workspace] Verified module: {filename}")

# 3. CRITICAL: Add current directory to Python sys.path so notebook cells can import data_utils, model, etc.
current_dir = os.path.abspath(os.getcwd())
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"\nActive working directory: {current_dir}")
print(f"Modules available: {[f for f in os.listdir(current_dir) if f.endswith(".py")]}")


## 3. Install Dependencies

In [ ]:
!pip install --quiet -r requirements.txt
print("Dependencies installed successfully!")

## 4. Optional: Weights & Biases (WandB) Setup
You can securely store your `WANDB_API_KEY` under Kaggle **Add-ons -> Secrets**.

In [ ]:
import os
USE_WANDB = False

try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key)
        print("[WandB] Successfully logged in via Kaggle Secrets!")
        USE_WANDB = True
    else:
        print("[WandB] WANDB_API_KEY secret is empty. Using local console logging.")
except Exception as e:
    print("[WandB] WandB secret not configured or offline mode. Running with local console logging.")

## 5. Dataset Discovery & Path Configuration
Scans `/kaggle/input/` for `.parquet` or `.csv` files. If none are attached yet, it activates `--synthetic` mode.

In [ ]:
import glob
from pathlib import Path

input_files = glob.glob("/kaggle/input/**/*", recursive=True)
parquet_files = [f for f in input_files if f.endswith(".parquet") or f.endswith(".csv")]

if parquet_files:
    data_path = parquet_files[0]
    synthetic_flag = ""
    print(f"[Data] Found dataset: {data_path}")
else:
    data_path = "/kaggle/working/data/synthetic_yambda.parquet"
    synthetic_flag = "--synthetic"
    print("[Data] No dataset found in /kaggle/input/. Using --synthetic generator for demonstration.")

## 6. Launch Training with GPU Acceleration & FP16
Executes `train_base.py` on the GPU with `--fp16`, `--split_mode global_temporal`, and evaluation on **HitRate@10** and **NDCG@10**.

In [ ]:
wandb_arg = "--use_wandb" if USE_WANDB else ""

!python train_base.py \
    --data_path {data_path} \
    --processed_dir /kaggle/working/data/processed \
    --output_dir /kaggle/working/checkpoints \
    --min_item_freq 5 \
    --max_length 64 \
    --n_embd 128 \
    --n_layer 3 \
    --n_head 4 \
    --batch_size 128 \
    --lr 5e-4 \
    --epochs 10 \
    --split_mode global_temporal \
    --fp16 \
    --num_workers 2 \
    {synthetic_flag} \
    {wandb_arg}

## 7. Model Verification & Top-K Recommendation Inference
Loads the best model checkpoint and tests next-item prediction on a user trajectory.

In [ ]:
import os
import sys
import torch

# Ensure current working directory is on sys.path
current_dir = os.path.abspath(os.getcwd())
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from data_utils import ItemTokenizer
from model import build_gpt2_recsys_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer_path = "/kaggle/working/data/processed/item_tokenizer.json"
ckpt_path = "/kaggle/working/checkpoints/best_gpt2_recsys.pt"

if os.path.exists(tokenizer_path) and os.path.exists(ckpt_path):
    tokenizer = ItemTokenizer.load(tokenizer_path)
    checkpoint = torch.load(ckpt_path, map_location=device)

    cfg = checkpoint["model_config"]
    model = build_gpt2_recsys_model(**cfg).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded Best Checkpoint (Epoch {checkpoint['epoch']}): Val HR@10 = {checkpoint['val_hr10']:.4f}, Val NDCG@10 = {checkpoint['val_ndcg10']:.4f}")

    # Demonstration: Sample next-item recommendation
    sample_input = torch.tensor([[1, 2, 3, 4]], dtype=torch.long, device=device)
    scores, topk = model.predict_next_items(sample_input, top_k=5)
    rec_items = tokenizer.inverse_transform(topk[0].cpu().numpy())
    print(f"\nInput History Tokens: {sample_input[0].tolist()}")
    print(f"Top-5 Recommended Next Tracks: {rec_items}")
else:
    print(f"Checkpoint or tokenizer not found. Make sure Cell 6 (training) has completed.")

## 8. Export Trained Artifacts
Compresses checkpoints and processed tokenizer mappings into a `.zip` archive for one-click download from `/kaggle/working`.

In [ ]:
!zip -r /kaggle/working/recsys_checkpoints.zip /kaggle/working/checkpoints /kaggle/working/data/processed
print("Checkpoints successfully archived to: /kaggle/working/recsys_checkpoints.zip")